In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
sys.path.append(".")

for pkg in ["agents", "tools", "state", "graph"]:
    os.makedirs(pkg, exist_ok=True)
    init = os.path.join(pkg, "__init__.py")
    if not os.path.exists(init):
        open(init, "w").close()
print("package __init__ files ready")

package __init__ files ready


In [7]:
%%writefile agents/web_search_agent.py
import time
from typing import Dict, List

from tools.llm_clients import research_with_fallback
from tools.logger import log_agent_event

AGENT_NAME = "web_search_agent"
MAX_QUERIES = 5  # caps Gemini grounding usage (free-tier quota) per run


def _dedupe_sources(sources: List[Dict]) -> List[Dict]:
    seen, unique = set(), []
    for s in sources:
        url = s.get("url")
        if url and url not in seen:
            seen.add(url)
            unique.append(s)
    return unique


def search_one(sub_question: str) -> Dict:
    """Research a single sub-question. Never raises: failures come back as
    status='failed' so one bad query can't crash the whole run."""
    start = time.time()
    try:
        raw = research_with_fallback(sub_question)
        sources = _dedupe_sources(raw.get("sources", []))
        answer = (raw.get("answer") or "").strip()
        status = "ok" if (answer and sources) else "empty"
        result = {
            "sub_question": sub_question,
            "answer": answer,
            "sources": sources,
            "provider": raw.get("provider", "unknown"),
            "status": status,
            "latency_s": round(time.time() - start, 2),
        }
    except Exception as e:
        result = {
            "sub_question": sub_question,
            "answer": "",
            "sources": [],
            "provider": "none",
            "status": "failed",
            "error": f"{type(e).__name__}: {e}",
            "latency_s": round(time.time() - start, 2),
        }
    log_agent_event(AGENT_NAME, "search_done", status=result["status"],
                    provider=result["provider"], sources=len(result["sources"]))
    return result


def web_search_node(state: Dict) -> Dict:
    """LangGraph node: researches the plan's sub-questions and writes to state."""
    seen, queries = set(), []
    for q in state.get("plan", []):
        q = q.strip()
        if q and q.lower() not in seen:
            seen.add(q.lower())
            queries.append(q)
    queries = queries[:MAX_QUERIES]

    if not queries:
        return {"errors": [{"agent": AGENT_NAME, "error": "empty plan: nothing to search"}]}

    results = [search_one(q) for q in queries]

    good = [r for r in results if r["status"] != "failed"]
    failed = [r for r in results if r["status"] == "failed"]

    usage: Dict[str, int] = {}
    for r in good:
        usage[r["provider"]] = usage.get(r["provider"], 0) + 1

    return {
        "search_results": good,
        "errors": [{"agent": AGENT_NAME, "sub_question": r["sub_question"], "error": r["error"]}
                   for r in failed],
        "model_usage_log": [{"agent": AGENT_NAME, "provider": p, "queries": n}
                            for p, n in usage.items()],
    }

Overwriting agents/web_search_agent.py


In [8]:
from agents.web_search_agent import search_one

r = search_one("What are the main safety risks of solid-state EV batteries?")
print("status:", r["status"], "| provider:", r["provider"], "| latency:", r["latency_s"], "s")
print("answer:", r["answer"][:250], "...")
for s in r["sources"][:3]:
    print("-", s["title"], "|", s["url"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:27:08,137 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


status: ok | provider: tavily | latency: 2.39 s
answer: For example, continuous charging and discharging cycles lead to an inevitable and gradual decline in battery capacity (energy storage), a process accelerated by frequent rapid charging. Liquid electrolytes also introduce risks such as leakage and den ...
- Challenges in Solid-State EV Batteries | https://www.batterypowertips.com/what-are-the-main-challenges-in-developing-solid-state-batteries-for-evs
- Solid-State Batteries: Pros, Cons, Applications & Testing Challenges | https://www.sinexcel-re.com/blog/solid-state-batteries-pros-cons-advantages-disadvantages
- Solid-State vs. Lithium-Ion Battery Safety | Anern | https://www.anernstore.com/blogs/diy-solar-guides/solid-state-battery-safety-lithium


In [9]:
test_queries = [
    "Tesla 4680 battery production status 2026",
    "How does lithium iron phosphate differ from NMC chemistry?",
    "CATL vs BYD global EV battery market share",
    "asdfgh qwerty zxcv",
    "sodium-ion battery commercialization timeline",
]

for q in test_queries:
    r = search_one(q)
    print(f"[{r['status']:6}] [{r['provider']:6}] {len(r['sources'])} sources | {q}")

Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:29:09,442 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


[ok    ] [tavily] 3 sources | Tesla 4680 battery production status 2026
Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:29:10,884 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


[ok    ] [tavily] 3 sources | How does lithium iron phosphate differ from NMC chemistry?
Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:29:13,814 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


[ok    ] [tavily] 3 sources | CATL vs BYD global EV battery market share
Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:29:16,025 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


[ok    ] [tavily] 3 sources | asdfgh qwerty zxcv
Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:29:18,040 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


[ok    ] [tavily] 3 sources | sodium-ion battery commercialization timeline


In [10]:
r = search_one("asdfgh qwerty zxcv")
for s in r["sources"]:
    print("-", s["title"], "|", s["url"])

Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:32:51,828 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


- Solved: Find the next term in the following sequence. qwerty, asdfgh , ? zbxvnc zvcxbn znxbcv z [Math] | https://www.gauthmath.com/solution/ZLwGvE5kF4U/Find-the-next-term-in-the-following-sequence-qwerty-asdfgh-zbxvnc-zvcxbn-znxbcv-
- Qwerty Informatica Information | https://rocketreach.co/qwerty-informatica-profile_b5c7e0aaf42e0dbe
- qwertyqwerty - Typing Practice | 10FastFingers | https://10fastfingers.com/text/6182-qwertyqwerty


In [11]:
from agents.web_search_agent import web_search_node

state = {
    "query": "Compare EV battery strategies of Tesla, BYD, and CATL",
    "plan": [
        "What battery chemistries do Tesla, BYD and CATL use?",
        "What battery chemistries do Tesla, BYD and CATL use?",   
        "How do their battery costs compare?",
        "",                                                        
    ],
}

out = web_search_node(state)
print("results:", len(out["search_results"]), "| errors:", len(out["errors"]))
print("usage:", out["model_usage_log"])
for r in out["search_results"]:
    print("-", r["sub_question"], "->", r["status"], f"({len(r['sources'])} sources)")

Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:36:59,391 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


Gemini unavailable (ClientError), falling back to Tavily...


2026-09-29 08:37:01,276 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=3


results: 2 | errors: 0
usage: [{'agent': 'web_search_agent', 'provider': 'tavily', 'queries': 2}]
- What battery chemistries do Tesla, BYD and CATL use? -> ok (3 sources)
- How do their battery costs compare? -> ok (3 sources)


In [12]:
%%writefile tests/test_web_search_agent.py
import agents.web_search_agent as wsa


def fake_ok(q):
    return {"answer": "some answer", "sources": [{"title": "T", "url": "http://a"}], "provider": "gemini"}


def fake_boom(q):
    raise RuntimeError("both providers down")


def test_search_one_ok(monkeypatch):
    monkeypatch.setattr(wsa, "research_with_fallback", fake_ok)
    r = wsa.search_one("q")
    assert r["status"] == "ok" and r["provider"] == "gemini"


def test_search_one_never_raises(monkeypatch):
    monkeypatch.setattr(wsa, "research_with_fallback", fake_boom)
    r = wsa.search_one("q")
    assert r["status"] == "failed" and "RuntimeError" in r["error"]


def test_search_one_empty_when_no_sources(monkeypatch):
    monkeypatch.setattr(wsa, "research_with_fallback",
                        lambda q: {"answer": "x", "sources": [], "provider": "tavily"})
    assert wsa.search_one("q")["status"] == "empty"


def test_node_dedupes_and_caps(monkeypatch):
    monkeypatch.setattr(wsa, "research_with_fallback", fake_ok)
    plan = ["a", "A", "b", "c", "d", "e", "f", "g"]
    out = wsa.web_search_node({"plan": plan})
    assert len(out["search_results"]) == wsa.MAX_QUERIES


def test_node_empty_plan(monkeypatch):
    out = wsa.web_search_node({"plan": []})
    assert out["errors"] and "search_results" not in out


def test_node_partial_failure(monkeypatch):
    def flaky(q):
        if q == "bad":
            raise RuntimeError("x")
        return fake_ok(q)
    monkeypatch.setattr(wsa, "research_with_fallback", flaky)
    out = wsa.web_search_node({"plan": ["good", "bad"]})
    assert len(out["search_results"]) == 1 and len(out["errors"]) == 1

Writing tests/test_web_search_agent.py


In [ ]:
with open("requirements.txt", "a") as f:
    f.write("pytest\n")
print(" pytest added to requirements.txt")